## Visualize Quality Metrics

Basically we just need to write some scaffolding to download all the metrics from CSD3 and then turn this into a pretty dictionary / table (you do you), to easily visualize. Easy peasy!

In [58]:
import os
import json
from pathlib import Path
from typing import Any

from huggingface_hub import snapshot_download
import matplotlib.pyplot as plt
import pandas as pd

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [65]:
def download_hf_files(repo_id: str, *, directory_path: str, local_path: Path):
    """Download files from a Hugging Face repository to a local directory."""
    if not local_path.exists():
        local_path.mkdir(parents=True, exist_ok=True)
    data_path = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=local_path,
        allow_patterns=[directory_path + "/*"],
    )
    return Path(data_path) / directory_path

In [66]:
metric_files = download_hf_files(
    "ljvmiranda921/mtep-intrinsic-metrics",
    directory_path="csd3",
    local_path=Path("./data/"),
)

Fetching 36 files: 100%|██████████| 36/36 [00:03<00:00, 11.56it/s]


In [67]:
def parse_metrics(metrics_dir: Path) -> list[dict]:
    """Parse metric files and extract language, model, and data.

    Expected filename format: msde-S1-{language}_{model}_intrinsic_metrics
    Example: msde-S1-ar_mistralai__Mistral-Small-24B-Instruct-2501_intrinsic_metrics
    """
    metrics_data = []
    for file in metrics_dir.glob("*.json"):
        filename = file.stem
        parts = filename.replace("msde-S1-", "").replace("_intrinsic_metrics", "")
        language, model_raw = parts.split("_", 1)
        model = model_raw.replace("__", "/")

        with open(file, "r") as f:
            data = json.load(f)

        metrics_data.append(
            # fmt: off
            {
                "language": language,
                "model": model,
                "prompts_distinct_ri": data.get("distinct_ri", {}).get("prompts_distinct_ri"),
                "responses_distinct_ri": data.get("distinct_ri", {}).get("prompts_distinct_ri"),
                "rubric_score": data.get("reward_model", {}).get("average_rubric_score"),
                "perplexity": data.get("perplexity", {}).get("average_perplexity"),
            }
            # fmt: on
        )

    return metrics_data

In [68]:
df = (
    pd.DataFrame(parse_metrics(metric_files))
    .sort_values(by=["language", "model"])
    .reset_index(drop=True)
)

In [69]:
df

,language,model,prompts_distinct_ri,responses_distinct_ri,rubric_score,perplexity
0,ar,CohereLabs/aya-expanse-32b,0.692789,0.692789,3.9640,None
1,ar,cohere-command-a,0.690184,0.690184,3.9956,None
2,ar,google/gemma-3-12b-it,0.721363,0.721363,3.7736,None
3,ar,google/gemma-3-27b-it,NaN,NaN,3.9315,None
4,ar,google/gemma-3-4b-it,NaN,NaN,3.4699,None
5,ar,gpt-4o-mini-2024-07-18,0.703685,0.703685,3.5159,None
6,ar,ibm-granite/granite-4.0-1b,NaN,NaN,2.4633,None
7,ar,ibm-granite/granite-4.0-micro,0.741429,0.741429,3.0330,None
8,ar,meta-llama/Llama-3.1-70B-Instruct,0.700566,0.700566,2.7185,None
9,ar,meta-llama/Llama-3.1-8B-Instruct,0.707724,0.707724,1.7314,None
